In [ ]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path

In [31]:
processed_directory = Path("/Users/jininaljayyousi/mindclick/data/processed")

epochs = mne.read_epochs(processed_directory / "p300_epochs.fif", preload=True)
labels = np.load(processed_directory / "p300_labels.npy")

print(epochs)
print(labels.shape)
print(np.unique(labels))

Reading /Users/jininaljayyousi/mindclick/data/processed/p300_epochs.fif ...
Isotrak not found
    Found the data of interest:
        t =    -199.22 ...     800.78 ms
        0 CTF compensation matrices available
Not setting metadata
540 matching events found
No baseline correction applied
0 projection items activated
<EpochsFIF | 540 events (all good), -0.199 – 0.801 s (baseline -0.199 – 0 s), ~17.0 MiB, data loaded,
 '1': 540>
(540,)
[0 1]


/var/folders/1k/m00ckqlx5cn3hklkwvwxq6wc0000gn/T/ipykernel_48192/1251296800.py:3: RuntimeWarning: This filename (/Users/jininaljayyousi/mindclick/data/processed/p300_epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(processed_directory / "p300_epochs.fif", preload=True)


In [46]:
channels_of_interest = [
    "EEG_Cz",
    "EEG_Pz",
    "EEG_P3",
    "EEG_CP3",
    "EEG_CP4",
    "EEG_PO7",
    "EEG_PO8",
    "EEG_Oz",
]

epochs_features = epochs.copy().pick(channels_of_interest)
epochs_features = epochs_features.crop(tmin=0.0, tmax=0.8)

X = epochs_features.get_data()
y = labels

print("Before flattening:", X.shape, X.ndim)

n_samples, n_channels, n_times = X.shape
X = X.reshape(n_samples, n_channels * n_times)

print("After flattening:", X.shape, X.ndim)

Before flattening: (540, 8, 206) 3
After flattening: (540, 1648) 2


In [47]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=23,
    stratify=y,
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced")
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.89      0.91        96
           1       0.35      0.50      0.41        12

    accuracy                           0.84       108
   macro avg       0.64      0.69      0.66       108
weighted avg       0.87      0.84      0.85       108

[[85 11]
 [ 6  6]]


/Users/jininaljayyousi/mindclick/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/jininaljayyousi/mindclick/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/jininaljayyousi/mindclick/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/jininaljayyousi/mindclick/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:212: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/jininaljayyousi/mindclick/venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:212: RuntimeWarning: overflow encountered in mat